# 24 — Greedy MoE phase 4: final report
Resolve the winning capacity, bounded comparison-stage weight decay, expert architecture, and representation objective; reuse completed seed runs and write paired paper-facing reports. Weight decay here is not final-model HPO.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER='selimsidan'; GITHUB_REPO='dataset_moe_nids'; GITHUB_BRANCH='main'; GITHUB_SECRET_NAME='GITHUB_TOKEN'
DRIVE_DATA_DIR='/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDY_CONFIG='config/greedy_moe_study_v2.yaml'; PREFIX='nfv3_4way_greedy_v2'
PHASE='4'; EXECUTE=False; MAX_NEW_RUNS=None; RUN_TESTS=True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive'); token=userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode(); git_env=os.environ|{'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.https://github.com/.extraheader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}'}
repo=Path('/content')/GITHUB_REPO; url=f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH] if (repo/'.git').is_dir() else ['git','clone','--branch',GITHUB_BRANCH,'--single-branch',url,str(repo)],env=git_env,check=True)
git_env.clear(); token=auth=None; os.chdir(repo)
os.environ.update(NIDS_DRIVE_BASE=DRIVE_DATA_DIR,NIDS_OUTPUT_DIR=DRIVE_OUTPUT_DIR,NIDS_SCRATCH_DIR='/content/dataset_moe_nids_scratch')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())

## Preflight, dry run, and final aggregation

In [ ]:
import torch
from training.ablation_matrix import expected_evaluated_run_count, load_study
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime; missing winner reports may require checkpoint evaluation.')
study=load_study(STUDY_CONFIG); print('GPU:',torch.cuda.get_device_name(0),'| planned runs:',expected_evaluated_run_count(study))
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q','tests/test_ablation_study.py'],check=True)
command=[sys.executable,'-u','-m','training.ablation_matrix','--study-config',STUDY_CONFIG,'--phase',PHASE,'--prefix',PREFIX]; subprocess.run(command,check=True,env=os.environ.copy())
if EXECUTE:
    execute_command=[*command,'--execute']
    if MAX_NEW_RUNS is not None: execute_command += ['--max-new-runs',str(MAX_NEW_RUNS)]
    subprocess.run(execute_command,check=True,env=os.environ.copy())

## Paper-facing outputs

In [ ]:
import pandas as pd
from IPython.display import display
summary_dir=Path(DRIVE_OUTPUT_DIR)/'results'/f'{PREFIX}_summary'; state_path=summary_dir/'study_state.json'
if state_path.is_file(): print(json.dumps(json.loads(state_path.read_text()).get('decisions',{}),indent=2))
for name in ['Final_Decision_Summary.csv','Final_Overall_Summary.csv','Final_Per_Dataset_Summary.csv','Final_Per_Class_Summary.csv','Final_Resource_Summary.csv']:
    path=summary_dir/name
    if path.is_file(): print(name); display(pd.read_csv(path))
manifest=summary_dir/'Final_Study_Manifest.json'
if manifest.is_file(): print(json.dumps(json.loads(manifest.read_text()),indent=2))
print('Final report directory:',summary_dir)

## Latent-space geometry, probes, and figures (final baseline and winner, all seeds)

In [ ]:
manifest_path = summary_dir / 'Final_Study_Manifest.json'
if manifest_path.is_file():
    from IPython.display import Image
    final_manifest = json.loads(manifest_path.read_text())
    final_run_ids = set(final_manifest['baseline_run_ids']) | set(final_manifest['winner_run_ids'])
    for name in [
        'Final_Latent_Snapshot_Summary.csv', 'Final_Latent_Per_Class_Summary.csv',
        'Final_Latent_Probe_Summary.csv', 'Final_Latent_Probe_Per_Class_Summary.csv',
    ]:
        path = summary_dir / name
        if path.is_file():
            print(name); display(pd.read_csv(path))
    snapshot_path = summary_dir / 'Latent_Snapshot_Metrics_By_Run.csv'
    if snapshot_path.is_file():
        snapshots = pd.read_csv(snapshot_path).query('run_id in @final_run_ids')
        print('Final baseline/winner per-seed latent snapshots:'); display(snapshots)
        for figure_dir in sorted(snapshots['latent_figures_dir'].dropna().unique()):
            for png in sorted(Path(figure_dir).glob('*.png')):
                print(png.name); display(Image(filename=str(png)))
else:
    print('Final study manifest is not available yet:', manifest_path)